In [2]:
import csv
import json
import os

def convert_csv_to_jsonl(input_file, output_file):
    PROMPT_TEMPLATE = (
        "Analyze the image carefully.\n\n"
        "Respond ONLY in JSON:\n"
        "{\n"
        "  \"unrealistic\": \"yes | no | somewhat\",\n"
        "  \"explanation\": \"string\"\n"
        "}\n\n"
        "Rules:\n"
        "- Use \"yes\" only for clear artifacts\n"
        "- Use \"somewhat\" if ambiguous\n"
        "- Use \"no\" if plausibly real\n"
        "- Explanation ≤ 30 words"
    )
    dataset = []
    
    with open(input_file, mode='r', encoding='utf-8') as f:
        # Using DictReader to automatically map column headers to keys
        reader = csv.DictReader(f)
        
        for row in reader:
            # Construct the nested structure
            entry = {
                "image": f"{row['file_name']}",
                "prompt": PROMPT_TEMPLATE,
                "response": {
                    "unrealistic": row['unrealistic'].lower(),
                    "explanation": row['description']
                }
            }
            dataset.append(entry)

    # Write to JSONL (one JSON object per line)
    # with open(output_file, mode='w', encoding='utf-8') as f:
    #     for entry in dataset:
    #         json_line = json.dumps(entry)
    #         f.write(json_line + '\n')

        
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    # Now safely write the file
    with open(output_file, mode='w', encoding='utf-8') as f:
        for entry in dataset:
            f.write(json.dumps(entry) + '\n')


# Usage
output_file = r"/home/manem/Qwen-3VL-Testing/Chat_input/REALM_desc_test.jsonl"
input_file = r"/home/manem/Qwen-3VL-Testing/REALM-desc/test/image_descriptions_processed.csv"
convert_csv_to_jsonl(input_file, output_file)

In [3]:
import json
import os

def transform_data(input_jsonl, image_dir, prompt):
    transformed = []
    with open(input_jsonl, 'r') as f:
        for line in f:
            item = json.loads(line)
            # Construct the ChatML structure
            entry = {
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {"type": "image", "image": os.path.join(image_dir, item["image"])},
                            {"type": "text", "text": PROMPT}
                        ]
                    },
                    {
                        "role": "assistant",
                        "content": json.dumps(item["response"]) # Convert dict back to JSON string
                    }
                ]
            }
            transformed.append(entry)
    return transformed

if __name__ == "__main__":
    input_jsonl = r"/home/manem/Qwen-3VL-Testing/Chat_input/REALM_desc_test.jsonl"
    image_dir = r"/home/manem/Qwen-3VL-Testing/dataset/images/test_images"
    output_jsonl = r"/home/manem/Qwen-3VL-Testing/Chat_input/REALM_desc_test_chatml.jsonl"
    
    PROMPT = (
        "Is there anything unrealistic in this image? yes or no or somewhat, "
        "if yes or somewhat explain in maximum 30 words, please ensure to explain "
        "what looks unreal like if face is distorted, or transition between objects is not smooth."
        "Respond ONLY in the following JSON format:\n"
        "{\n"
        '  "unrealistic": "yes | no | somewhat",\n'
        '  "explanation": "string"\n'
        "}\n\n"
    )

    transformed_data = transform_data(input_jsonl, image_dir, PROMPT)

    os.makedirs(os.path.dirname(output_jsonl), exist_ok=True)
    with open(output_jsonl, 'w') as f:
        for item in transformed_data:
            f.write(json.dumps(item) + '\n')


In [4]:
import json
import os

file_path = r'/home/manem/Qwen-3VL-Testing/Chat_input/REALM_desc_test_chatml.jsonl'

# 1. Read the data into memory
processed_lines = []

with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        
        data = json.loads(line)
        
        # 2. Process each message in the list
        for message in data.get('messages', []):
            # Transform assistant string content into a list of objects
            if message.get('role') == 'assistant' and isinstance(message.get('content'), str):
                message['content'] = [
                    {
                        "type": "text",
                        "text": message['content']
                    }
                ]
        
        processed_lines.append(json.dumps(data))

# 3. Write back to the same file (overwriting)
with open(file_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(processed_lines) + '\n')

print(f"File '{file_path}' has been updated in-place to match Dataset-1 format.")

File '/home/manem/Qwen-3VL-Testing/Chat_input/REALM_desc_test_chatml.jsonl' has been updated in-place to match Dataset-1 format.
